In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def replace_na_with_mean(df):
    return df.fillna(df.mean())

def eda_target_description(df, target_column):
    print(f"\nEDA for dataset with target column '{target_column}':")
    print(df[target_column].describe())
    print("\nValue counts for target column:")
    print(df[target_column].value_counts())
    sns.countplot(data=df, x=target_column)
    plt.show()

def split_and_cross_validate(df, target_column):
    df = replace_na_with_mean(df)  
    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Print shapes of training and testing sets
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    model = RandomForestClassifier(random_state=42)

    skf = StratifiedKFold(n_splits=5)

    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
    print(f"Cross-validation accuracy scores: {scores}")
    print(f"Mean cross-validation accuracy: {scores.mean()}")

def run_regressions_and_visualize(df, target_col):
    df = replace_na_with_mean(df)  

    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Print shapes of training and testing sets
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)
    y_pred_linear = linear_model.predict(X_test)

    tree_model = DecisionTreeRegressor(random_state=42)
    tree_model.fit(X_train, y_train)
    y_pred_tree = tree_model.predict(X_test)

    voting_model = VotingRegressor([('lr', linear_model), ('tree', tree_model)])
    voting_model.fit(X_train, y_train)
    y_pred_voting = voting_model.predict(X_test)

    xgb_model = XGBRegressor(random_state=42)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    catboost_model = CatBoostRegressor(silent=True, random_state=42)
    catboost_model.fit(X_train, y_train)
    y_pred_catboost = catboost_model.predict(X_test)

    print('Performance Metrics:')
    for name, y_pred in zip(['Linear', 'Decision Tree', 'Voting', 'XGBoost', 'CatBoost'], 
                             [y_pred_linear, y_pred_tree, y_pred_voting, y_pred_xgb, y_pred_catboost]):
        mse = mean_squared_error(y_test, y_pred)
        rmse = mse ** 0.5
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  

        print(f"{name} Regressor: MSE = {mse:.2f}, RMSE = {rmse:.2f}, R^2 = {r2:.2f}, MAE = {mae:.2f}, MAPE = {mape:.2f}%")

datasets_cq = {
    'default': r"C:\Users\vvmad\Downloads\5th\ML\PROJECT\datasets\java_cq_embed_data.csv",
    'pca': r"C:\Users\vvmad\Downloads\5th\ML\PROJECT\datasets\reduced-cq\cq_pca_reduced.csv",
    'tsne': r"C:\Users\vvmad\Downloads\5th\ML\PROJECT\datasets\reduced-cq\cq_tsne_reduced.csv",
    'umap': r"C:\Users\vvmad\Downloads\5th\ML\PROJECT\datasets\reduced-cq\cq_umap_reduced.csv"
}

target_column = 'Final_Marks'  

for name, path in datasets_cq.items():
    df = pd.read_csv(path)
    
    print(f"\n--- EDA for {name} dataset ---")
    eda_target_description(df, target_column)
    
    print(f"\n--- Cross-validation for {name} dataset ---")
    split_and_cross_validate(df, target_column)

    print(f"\n--- Regression for {name} dataset ---")
    run_regressions_and_visualize(df, target_column)
